

$$ max_{virtuais} = (n-1)\cdot d \cdot Q^{,d-1} $$


onde

* (n) = número de amostras reais,
* (d) = número de entradas (dimensões),
* (Q) = número de quantis (Q_a) usados (isto é, o número de valores de (a) que você fixa para as outras dimensões). 

Aplicando aos seus valores:

* (n=25) (25 amostras) → (n-1=24)
* (d=10) (10 entradas)
* se usar o conjunto sugerido ({0.025,0.25,0.5,0.75,0.975}) → (Q=5)


$$ max_{virtuais} = 24 \times 10 \times 5^{9} = 468,750,000 $$


Ou seja, **468 750 000** amostras virtuais no máximo (explosão combinatória enorme).

Observações práticas importantes

* Esse número é teórico — na prática é inviável computacionalmente e muitas dessas combinações podem ser irrelevantes ou redundantes.
* Os autores mencionam justamente essa explosão e, em aplicações reais, costumam reduzir (Q) (por exemplo, usar somente (a=0.5), i.e. (Q=1)) para diminuir o custo. Com (Q=1) o mesmo cálculo dá (24\times 10 \times 1^{9}=240) (ordem de centenas), muito mais manejável — e coincide com a motivação do artigo para escolher menos quantis em problemas reais. 
* Outro ponto: o número de saídas (1 saída no seu caso) **não** entra na fórmula — só importam entradas/dimensões, quantis e (n).


In [1]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, ExpSineSquared, DotProduct, WhiteKernel, ConstantKernel as C
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
out_scaler = StandardScaler()

In [2]:

PREDICTORS = ["pH", "Cond", "Temp", "OD", "Tds", "Resist", "Salin", "ORP", "IP", "Cor"] # 10 entradas
PREDICTORS = ["pH", "Cond", "Temp", "OD", "Tds", "Fe"] # 10 entradas

TARGETS = ["Fe", "Al", "As", "Pb", "Zn", "Hg", "Co", "V", "Ba", "Mn"] # 10 saidas

In [3]:
Data = pd.read_excel("./Dados/Dados.xlsx")

Datasets = []

for n in range(1, 5):
    n_data = Data[Data["Pontos"] == f"P{n}" ]
    n_data = n_data.drop(columns=["Campanhas", "Pontos"])

    Datasets.append(n_data)

    

# Métricas

In [4]:
import matplotlib.pyplot as plt
import numpy as np
import os

def PlotPredictions(train_orig, train_pred, test_orig, test_pred, target_name, n): 
    # cria figura
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # -----------------------------
    # SUBPLOT 1 — TREINO
    # -----------------------------
    ax = axes[0]
    n_train = len(train_orig)
    x_train = np.arange(n_train)

    ax.plot(x_train, train_orig, label="Original (train)", color="blue", )
    ax.plot(x_train, train_pred, label="Predito (train)", color="red",)

    ax.scatter(x_train, train_orig, color="blue", s=35)
    ax.scatter(x_train, train_pred, color="red", s=35)

    ax.set_title("Treinamento")
    ax.set_xlabel("Amostras")
    ax.set_ylabel(target_name)
    ax.grid(True)
    ax.legend()

    # -----------------------------
    # SUBPLOT 2 — TESTE
    # -----------------------------
    ax = axes[1]
    n_test = len(test_orig)
    x_test = np.arange(n_test)

    ax.plot(x_test, test_orig, label="Original (test)", color="blue", linewidth=1.8)
    ax.plot(x_test, test_pred, label="Predito (test)", color="red", linewidth=1.8)

    ax.scatter(x_test, test_orig, color="blue", s=35)
    ax.scatter(x_test, test_pred, color="red", s=35)

    ax.set_title("Teste")
    ax.set_xlabel("Amostras")
    ax.set_ylabel(target_name)
    ax.grid(True)
    ax.legend()

    plt.tight_layout()

    # salva a figura
    filename = f"./Dados/VirtualData/P{n}/TrainResults/{target_name}.pdf"
    plt.savefig(filename, format="pdf", bbox_inches="tight")

    # fecha a figura (IMPORTANTE para não acumular memória)
    plt.close(fig)

    print(f"Figura salva em: {filename}")


In [5]:
from sklearn.metrics import r2_score, mean_squared_error 

def ComputeMetrics(model, df_train, df_test, target, n):
    
    test_pred = out_scaler.inverse_transform(model.execute(df_test[PREDICTORS]).reshape(-1, 1))
    train_pred = out_scaler.inverse_transform(model.execute(df_train[PREDICTORS]).reshape(-1, 1))
    
    test_orig = out_scaler.inverse_transform(df_test[[target]])
    train_orig = out_scaler.inverse_transform(df_train[[target]])
    
    PlotPredictions(train_orig, train_pred, test_orig, test_pred, target, n)
    return {
        'r2_test': r2_score(test_orig, test_pred),
        'mse_test': mean_squared_error(test_orig, test_pred),
        'r2_train': r2_score(train_orig, train_pred),
        'mse_train': mean_squared_error(train_orig, train_pred),
    }

# Extraindo dados

In [6]:
Krbf = C(1.0) * RBF(length_scale=1.0, length_scale_bounds=(1e-9, 1e+6)) # R2 < 0
Kmtrn = C(1.0) * Matern(length_scale=1.0, nu=0.5, length_scale_bounds=(1e-9, 1e+6)) # R2 < 0
Krq = C(1.0) * RationalQuadratic(length_scale=1.0, alpha=0.1, length_scale_bounds=(1e-9, 1e+6)) # 
Kess = C(1.0) * ExpSineSquared(length_scale=1.0, periodicity=3.0, length_scale_bounds=(1e-9, 1e+6)) 
Kdp = C(1.0) * DotProduct() + WhiteKernel()

In [7]:

def PrepareData(Dataset, target):
    scaler.fit(Dataset[PREDICTORS])
    out_scaler.fit(Dataset[[target]])
    
    TrainData, TestData = train_test_split(Dataset, test_size=0.2, random_state=42)
    NormTrainData, NormTestData = TrainData, TestData 
    
    NormTrainData[PREDICTORS] = scaler.transform(TrainData[PREDICTORS])
    NormTrainData[target] = out_scaler.transform(TrainData[[target]])
    
    NormTestData[PREDICTORS] = scaler.transform(TestData[PREDICTORS])
    NormTestData[target] = out_scaler.transform(TestData[[target]])
    
    return TrainData, NormTrainData, TestData, NormTestData


In [8]:
kernel =  Kmtrn

class GPRVSG:
    def __init__(self, Dataset, target):
        self.target = target
        self.TrainData, self.NormTrainData, self.TestData, self.NormTestData = PrepareData(Dataset,
                                                                                           target)
        self.x_train =  self.NormTrainData[PREDICTORS].values
        self.y_train = self.NormTrainData[target].values
        
        self.x_test = self.NormTestData[PREDICTORS].values
        self.y_test = self.NormTestData[target].values
        
        self.d = self.x_train.shape[1]
        self.virtual_samples_x = []
        self.virtual_samples_y = []

    def ComputeProjection(self):
        projections = []
        for m in range(self.d):
            x_m = self.x_train[:, m]         # valores reais da dimensão m
            s_m = np.sort(x_m)               # projeções ordenadas
            projections.append(s_m)
        return projections


    def SetInputSpace(self):
        projections = self.ComputeProjection()
        avg_dists = [np.mean(np.diff(proj)) for proj in projections]

        Q_alpha = np.quantile(projections, [0.5], axis=1, method="hazen").T
        for m in range(self.d):
            for i in range(len(projections[m]) - 1):
                dist = projections[m][i + 1] - projections[m][i]
                if dist > avg_dists[m]:
                    G = (projections[m][i] + projections[m][i + 1]) / 2
                    for q in range(self.d):
                        if q != m:
                            for quantile in Q_alpha[q]:
                                tilde_q = np.zeros(self.d)
                                tilde_q[m] = G
                                tilde_q[q] = quantile
                                self.virtual_samples_x.append(tilde_q)
        self.virtual_samples_x = np.array(self.virtual_samples_x)

    def BuildModel(self):
        self.model = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=100,
                                              alpha=1e-8, normalize_y=True)
        self.model.fit(self.x_train, self.y_train)

    def getX(self):
        n_dims = self.virtual_samples_x.shape[1]
        col_names = [f"x{i+1}" for i in range(n_dims)]
        df = pd.DataFrame(self.virtual_samples_x, columns=col_names)
        return [df[col] for col in col_names]

    def ComputeY(self,):
        # pega lista de colunas: [x1, x2, ..., xn]
        X_cols = self.getX()

        # monta matriz X de entrada
        X = np.column_stack(X_cols)

        # prediz y
        self.virtual_samples_y = self.model.predict(X)

        # cria nomes x1, x2, ..., xn
        n_dims = len(X_cols)

        # monta DataFrame final
        data = {name: X_cols[i] for i, name in enumerate(PREDICTORS)}
        data[self.target] = self.virtual_samples_y

        self.virtual_samples_df = pd.DataFrame(data)

    def execute(self, *coords):
        # transformar lista de vetores em matriz X
        X = np.column_stack(coords)

        y_pred = self.model.predict(X)
        return y_pred

    def Run(self):
        self.SetInputSpace()
        self.BuildModel()
        self.ComputeY()

In [9]:
def PlotVirtualData(virtual_df, original_df, predictors, target, n):
    savepath = f"./Dados/VirtualData/P{n+1}/VSGResults/Virtual_{target}.pdf"

    # total = 10 entradas + 1 saída
    total_features = len(predictors) + 1
    fig, axes = plt.subplots(total_features, 1, figsize=(10, 2*total_features), sharex=False)

    # junta entradas + saída
    features = predictors + [target]

    for i, feat in enumerate(features):

        ax = axes[i]

        # valores preditos (virtuais)
        y_pred = virtual_df[feat].values
        x_pred = np.arange(len(y_pred))

        # valores originais reais (Dataset)
        y_orig = original_df[feat].values
        x_orig = np.arange(len(y_orig))

        # plot
        ax.plot(x_pred, y_pred, color="red", label="Virtual (predito)", linewidth=1.7)
        ax.scatter(x_pred, y_pred, color="red", s=20)

        ax.plot(x_orig, y_orig, color="blue", label="Original", linewidth=1.7)
        ax.scatter(x_orig, y_orig, color="blue", s=20)

        ax.set_ylabel(feat)
        ax.grid(True)

        if i == 0:
            ax.legend()

    axes[-1].set_xlabel("Amostras")

    plt.tight_layout()
    plt.savefig(savepath, format="pdf", bbox_inches="tight")
    plt.close(fig)

    print(f"Figura salva em: {savepath}")


In [10]:

import os

def GetVirtualData():
    for n in range(0, 4):  
        metrics_all = []
        output_dir = f"./Dados/VirtualData/P{n+1}"
        os.makedirs(output_dir, exist_ok=True)
        os.makedirs(f"./Dados/VirtualData/P{n+1}/TrainResults/", exist_ok=True)
        os.makedirs(f"./Dados/VirtualData/P{n+1}/VSGResults/", exist_ok=True)

        for target in TARGETS:            
            gpr_vsg = GPRVSG(Datasets[n], target)
            gpr_vsg.Run()
            metrics = ComputeMetrics(gpr_vsg,gpr_vsg.NormTrainData,gpr_vsg.NormTestData,target,n+1)

            vs_filename = os.path.join(output_dir, f"virtual_samples_{target}.xlsx")
            gpr_vsg.virtual_samples_df.to_excel(vs_filename, index=False)

            # Guarda métricas + nome do target
            metrics_all.append(metrics)
            metrics["target"] = target  
            
            PlotVirtualData(
                virtual_df = gpr_vsg.virtual_samples_df,          # dados virtuais desnormalizados
                original_df = Datasets[n],                        # dataset original desnormalizado
                predictors = PREDICTORS,                          # entradas
                target = target,                                  # saída
                n = n
            )
            # break
        # Converte lista de dicts para DataFrame
        df_metrics = pd.DataFrame(metrics_all)
        display(df_metrics)

        # Salva arquivo final por ponto
        metrics_filename = os.path.join(output_dir, "GprMetrics.xlsx")
        df_metrics.to_excel(metrics_filename, index=False)


In [11]:
GetVirtualData()

Figura salva em: ./Dados/VirtualData/P1/TrainResults/Fe.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Fe.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/Al.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Al.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/As.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_As.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/Pb.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Pb.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/Zn.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Zn.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/Hg.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Hg.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/Co.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_Co.pdf
Figura salva em: ./Dados/VirtualData/P1/TrainResults/V.pdf
Figura salva em: ./Dados/VirtualData/P1/VSGResults/Virtual_

,r2_test,mse_test,r2_train,mse_train,target
0,0.857360,0.114614,1.0,2.027562e-16,Fe
1,-19.895800,8613.775649,1.0,8.478927e-13,Al
2,-0.224216,0.000607,1.0,1.662110e-19,As
3,-33.230978,0.336345,1.0,5.059785e-17,Pb
4,0.008609,508.390931,1.0,7.699294e-14,Zn
5,-13.105749,0.003029,1.0,1.720050e-18,Hg
6,-0.107912,0.073552,1.0,7.686723e-18,Co
7,0.184412,0.036468,1.0,9.162125e-18,V
8,0.439555,31.624060,1.0,1.727225e-14,Ba
9,-0.345116,4480.529646,1.0,3.704219e-13,Mn


Figura salva em: ./Dados/VirtualData/P2/TrainResults/Fe.pdf
Figura salva em: ./Dados/VirtualData/P2/VSGResults/Virtual_Fe.pdf
Figura salva em: ./Dados/VirtualData/P2/TrainResults/Al.pdf
Figura salva em: ./Dados/VirtualData/P2/VSGResults/Virtual_Al.pdf
Figura salva em: ./Dados/VirtualData/P2/TrainResults/As.pdf
Figura salva em: ./Dados/VirtualData/P2/VSGResults/Virtual_As.pdf
Figura salva em: ./Dados/VirtualData/P2/TrainResults/Pb.pdf
Figura salva em: ./Dados/VirtualData/P2/VSGResults/Virtual_Pb.pdf
Figura salva em: ./Dados/VirtualData/P2/TrainResults/Zn.pdf
Figura salva em: ./Dados/VirtualData/P2/VSGResults/Virtual_Zn.pdf
Figura salva em: ./Dados/VirtualData/P2/TrainResults/Hg.pdf
Figura salva em: ./Dados/VirtualData/P2/VSGResults/Virtual_Hg.pdf
Figura salva em: ./Dados/VirtualData/P2/TrainResults/Co.pdf
Figura salva em: ./Dados/VirtualData/P2/VSGResults/Virtual_Co.pdf
Figura salva em: ./Dados/VirtualData/P2/TrainResults/V.pdf
Figura salva em: ./Dados/VirtualData/P2/VSGResults/Virtual_

,r2_test,mse_test,r2_train,mse_train,target
0,0.545062,0.582057,1.0,1.801404e-16,Fe
1,-0.527183,5436.603732,1.0,7.093587e-13,Al
2,0.015424,0.000643,1.0,1.187287e-19,As
3,-0.301150,4.439459,1.0,5.429713e-17,Pb
4,-0.096002,562.581418,1.0,7.269468e-14,Zn
5,-8.606611,0.002124,1.0,6.836729e-19,Hg
6,0.278923,0.054591,1.0,1.693643e-17,Co
7,0.074402,0.039052,1.0,1.533123e-17,V
8,0.043931,48.895979,1.0,2.670539e-14,Ba
9,-0.259787,4954.054078,1.0,7.437658e-13,Mn


Figura salva em: ./Dados/VirtualData/P3/TrainResults/Fe.pdf
Figura salva em: ./Dados/VirtualData/P3/VSGResults/Virtual_Fe.pdf
Figura salva em: ./Dados/VirtualData/P3/TrainResults/Al.pdf
Figura salva em: ./Dados/VirtualData/P3/VSGResults/Virtual_Al.pdf
Figura salva em: ./Dados/VirtualData/P3/TrainResults/As.pdf
Figura salva em: ./Dados/VirtualData/P3/VSGResults/Virtual_As.pdf
Figura salva em: ./Dados/VirtualData/P3/TrainResults/Pb.pdf
Figura salva em: ./Dados/VirtualData/P3/VSGResults/Virtual_Pb.pdf
Figura salva em: ./Dados/VirtualData/P3/TrainResults/Zn.pdf
Figura salva em: ./Dados/VirtualData/P3/VSGResults/Virtual_Zn.pdf
Figura salva em: ./Dados/VirtualData/P3/TrainResults/Hg.pdf
Figura salva em: ./Dados/VirtualData/P3/VSGResults/Virtual_Hg.pdf
Figura salva em: ./Dados/VirtualData/P3/TrainResults/Co.pdf
Figura salva em: ./Dados/VirtualData/P3/VSGResults/Virtual_Co.pdf
Figura salva em: ./Dados/VirtualData/P3/TrainResults/V.pdf
Figura salva em: ./Dados/VirtualData/P3/VSGResults/Virtual_

,r2_test,mse_test,r2_train,mse_train,target
0,0.114514,1.372822,1.0,1.436660e-16,Fe
1,-19.070421,11588.817472,1.0,2.813790e-12,Al
2,0.441798,0.001326,1.0,3.757174e-19,As
3,-0.578145,0.060635,1.0,2.284149e-17,Pb
4,-0.026770,1022.595286,1.0,3.433376e-14,Zn
5,-44.640632,0.013094,1.0,1.450672e-17,Hg
6,0.219240,0.044190,1.0,8.162140e-18,Co
7,0.333253,0.034657,1.0,1.264510e-17,V
8,-0.067697,55.701365,1.0,1.688424e-14,Ba
9,0.306874,2016.343805,1.0,2.251871e-13,Mn


Figura salva em: ./Dados/VirtualData/P4/TrainResults/Fe.pdf
Figura salva em: ./Dados/VirtualData/P4/VSGResults/Virtual_Fe.pdf
Figura salva em: ./Dados/VirtualData/P4/TrainResults/Al.pdf
Figura salva em: ./Dados/VirtualData/P4/VSGResults/Virtual_Al.pdf
Figura salva em: ./Dados/VirtualData/P4/TrainResults/As.pdf
Figura salva em: ./Dados/VirtualData/P4/VSGResults/Virtual_As.pdf
Figura salva em: ./Dados/VirtualData/P4/TrainResults/Pb.pdf
Figura salva em: ./Dados/VirtualData/P4/VSGResults/Virtual_Pb.pdf
Figura salva em: ./Dados/VirtualData/P4/TrainResults/Zn.pdf
Figura salva em: ./Dados/VirtualData/P4/VSGResults/Virtual_Zn.pdf
Figura salva em: ./Dados/VirtualData/P4/TrainResults/Hg.pdf
Figura salva em: ./Dados/VirtualData/P4/VSGResults/Virtual_Hg.pdf
Figura salva em: ./Dados/VirtualData/P4/TrainResults/Co.pdf
Figura salva em: ./Dados/VirtualData/P4/VSGResults/Virtual_Co.pdf
Figura salva em: ./Dados/VirtualData/P4/TrainResults/V.pdf
Figura salva em: ./Dados/VirtualData/P4/VSGResults/Virtual_

,r2_test,mse_test,r2_train,mse_train,target
0,0.038920,1.389059,1.0,2.603060e-16,Fe
1,-3.571130,8974.915177,1.0,4.431681e-12,Al
2,0.293362,0.001353,1.0,1.233034e-18,As
3,-14.822276,0.181631,1.0,9.378834e-17,Pb
4,-0.090371,495.983545,1.0,3.954952e-14,Zn
5,-0.014174,0.007105,1.0,5.527087e-19,Hg
6,0.648670,0.007351,1.0,6.739852e-18,Co
7,0.338948,0.025119,1.0,1.524083e-17,V
8,0.491627,27.065927,1.0,7.378538e-14,Ba
9,0.608527,516.995734,1.0,2.243799e-13,Mn
